# Pretrained models

> Save world models with their config, and load them back from a cache folder or HuggingFace (stable-worldmodel API).

In [ ]:
#| default_exp wm.utils

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
# Adapted from stable-worldmodel's world models package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import json
import logging
import urllib.request
from pathlib import Path
from typing import Any

import torch

from stable_marl.data.dataset import get_cache_dir
from stable_marl.utils.core import HF_BASE_URL

logger = logging.getLogger(__name__)

In [ ]:
#| export
def save_pretrained(
    model: torch.nn.Module,
    run_name: str,
    config: dict | None = None,
    config_key: str | None = None,
    filename: str = 'weights.pt',
    cache_dir: str | Path | None = None) -> Path:
    """
    Save the weights of `model` to ``<cache_dir>/checkpoints/<run_name>/<filename>``, and
    `config` (``config[config_key]`` if given; a dict or an OmegaConf config with a Hydra
    ``_target_``) to ``config.json`` next to them, from which :func:`load_pretrained` rebuilds the
    model. Without `config`, the model's ``pretrained_config`` is used (e.g. a
    :class:`DecentralizedWorldModel` made with ``build_lewm``). Returns the weights' path.
    """
    ckpt_dir = get_cache_dir(cache_dir, sub_folder='checkpoints') / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = ckpt_dir / filename
    torch.save(model.state_dict(), checkpoint_path)

    if config is None:
        config = getattr(model, 'pretrained_config', None)
    if config is None:
        logger.warning('No config! Loading will have to be done manually.')
        return checkpoint_path
    if config_key is not None and config_key in config:
        config = config[config_key]
    from omegaconf import OmegaConf
    if OmegaConf.is_config(config):
        config = OmegaConf.to_container(config, resolve=True)
    with open(ckpt_dir / 'config.json', 'w') as f:
        json.dump(config, f, indent=2)
    logger.info(f'Model saved to {checkpoint_path}')
    return checkpoint_path

In [ ]:
#| export
def load_pretrained(name: str, cache_dir: str | Path | None = None, extra_args: dict | None = None) -> torch.nn.Module:
    """
    Load a model saved by :func:`save_pretrained`. `name` is, relative to ``<cache_dir>/checkpoints/``:

    1. a ``.pt`` file (its folder holds the ``config.json``): ``'my_run/weights_epoch_10.pt'``
    2. a folder with one ``.pt`` file and a ``config.json``: ``'my_run'``
    3. a HuggingFace repository ``<user>/<repo>`` (its ``config.json`` and ``weights.pt``), cached
       under ``models--<user>--<repo>``

    The model is built with Hydra's ``instantiate`` from the config, overridden by `extra_args`
    (dotted keys, e.g. ``{'dropout': 0.0}``), and the weights are loaded on the CPU.
    """
    from hydra.utils import instantiate
    cache_dir = get_cache_dir(cache_dir, sub_folder='checkpoints')
    checkpoint_path, config = _resolve(name, cache_dir)
    state_dict = torch.load(checkpoint_path, map_location='cpu')
    for key, value in (extra_args or {}).items():
        *parents, last = key.split('.')
        d = config
        for part in parents:
            d = d.setdefault(part, {})
        d[last] = value
    model = instantiate(config)
    model.load_state_dict(state_dict)
    return model


def _resolve(name: str, cache_dir: Path) -> tuple[Path, dict]:
    "``(checkpoint_path, config)`` of `name`: a .pt file, a folder, else a HuggingFace repo."
    local = cache_dir / name
    if local.suffix == '.pt':
        if not local.exists():
            raise FileNotFoundError(f'Checkpoint not found: {local}')
        return local, _load_config(local.parent)
    if local.is_dir() and list(local.glob('*.pt')):
        return _resolve_folder(local)
    if '/' in name:
        return _resolve_hf(name, cache_dir)
    raise ValueError(f"Cannot resolve '{name}': not a .pt file, a folder, or a HF repo id.")


def _resolve_folder(folder: Path) -> tuple[Path, dict]:
    pt_files = list(folder.glob('*.pt'))
    if not pt_files:
        raise FileNotFoundError(f'No .pt file found in {folder}')
    if len(pt_files) > 1:
        raise ValueError(f'Ambiguous checkpoint: multiple .pt files in {folder}. Specify the file directly.')
    return pt_files[0], _load_config(folder)


def _resolve_hf(repo_id: str, cache_dir: Path) -> tuple[Path, dict]:
    "A HuggingFace repo, downloaded once to ``<cache_dir>/models--<user>--<repo>/``."
    local_dir = cache_dir / f'models--{repo_id.replace("/", "--")}'
    if local_dir.is_dir():
        return _resolve_folder(local_dir)
    logger.info(f'Downloading {repo_id} from HuggingFace...')
    local_dir.mkdir(parents=True, exist_ok=True)
    for filename in ('config.json', 'weights.pt'):
        _download(f'{HF_BASE_URL}/{repo_id}/resolve/main/{filename}', local_dir / filename)
    return _resolve_folder(local_dir)


def _download(url: str, dest: Path):
    response = urllib.request.urlopen(url)
    try:
        from tqdm import tqdm
        bar = tqdm(total=int(response.headers.get('Content-Length', 0)) or None, unit='B', unit_scale=True, desc=dest.name)
    except ImportError:
        bar = None
    with open(dest, 'wb') as f:
        while chunk := response.read(8192):
            f.write(chunk)
            if bar is not None: bar.update(len(chunk))
    if bar is not None: bar.close()


def _load_config(folder: Path) -> dict:
    config_path = folder / 'config.json'
    if not config_path.exists():
        raise FileNotFoundError(f'config.json not found in {folder}')
    with open(config_path) as f:
        return json.load(f)

### Tests

In [ ]:
import tempfile, shutil
from fastcore.test import test_fail
from stable_marl.wm import DecentralizedWorldModel

TINY = dict(image_size=16, patch_size=8, embed_dim=16, depth=1, heads=2, dim_head=8, mlp_dim=32, projector_hidden=32,
            encoder_kwargs=dict(dim=16, depth=1, heads=2, mlp_dim=32), history_size=2, extra_inputs={'position': 2})
wm = DecentralizedWorldModel.build_lewm(num_agents=2, num_actions=4, **TINY).eval()
info = {'pixels': torch.randint(0, 256, (1, 1, 2, 2, 16, 16, 3), dtype=torch.uint8), 'position': torch.rand(1, 1, 2, 2, 2),
        'action_history': torch.randint(0, 4, (1, 1, 1, 2, 1)).float()}
cand = torch.nn.functional.one_hot(torch.randint(0, 4, (1, 1, 3, 2, 1)), 4).float().flatten(-2)
same = lambda m: torch.allclose(m.eval().rollout(dict(info), cand)['predicted_emb'], wm.rollout(dict(info), cand)['predicted_emb'])

with tempfile.TemporaryDirectory() as tmp:
    path = save_pretrained(wm, 'run', cache_dir=tmp)                 # config from the model
    assert path == Path(tmp) / 'checkpoints' / 'run' / 'weights.pt' and (path.parent / 'config.json').exists()
    assert same(load_pretrained('run', cache_dir=tmp))                # a folder
    assert same(load_pretrained('run/weights.pt', cache_dir=tmp))     # a .pt file
    # dotted overrides of the config (here, ones that keep the weights' shapes)
    loaded = load_pretrained('run', cache_dir=tmp, extra_args={'dropout': 0.25, 'encoder_kwargs.heads': 4})
    assert loaded.config['dropout'] == 0.25 and loaded.config['encoder_kwargs'] == {**TINY['encoder_kwargs'], 'heads': 4}

    # an OmegaConf config, under a key
    from omegaconf import OmegaConf
    save_pretrained(wm, 'run2', config=OmegaConf.create({'model': wm.pretrained_config, 'lr': 1}), config_key='model',
                    filename='epoch_3.pt', cache_dir=tmp)
    assert same(load_pretrained('run2/epoch_3.pt', cache_dir=tmp))
    shutil.copy(f'{tmp}/checkpoints/run2/epoch_3.pt', f'{tmp}/checkpoints/run2/epoch_4.pt')
    test_fail(lambda: load_pretrained('run2', cache_dir=tmp), contains='Ambiguous')
    test_fail(lambda: load_pretrained('nope', cache_dir=tmp), contains='Cannot resolve')
    test_fail(lambda: load_pretrained('run/missing.pt', cache_dir=tmp), contains='not found')

    # a HuggingFace repo already in the cache is not downloaded again
    shutil.copytree(f'{tmp}/checkpoints/run', f'{tmp}/checkpoints/models--someone--marl-wm')
    assert same(load_pretrained('someone/marl-wm', cache_dir=tmp))

    # without a config: weights only
    save_pretrained(torch.nn.Linear(2, 2), 'bare', cache_dir=tmp)
    assert not (Path(tmp) / 'checkpoints' / 'bare' / 'config.json').exists()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()